# Spillerstatistikk over tid

Velg sesong og spiller i nedtrekksmenyene. Notebooken bruker `cleaned_merged_seasons_team_aggregated_expanded.csv`, som bygges av `build_expanded_dataset.ipynb`.

In [ ]:
%pip install -q pandas matplotlib ipywidgets

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option("display.max_columns", 100)

candidates = [
    Path.cwd() / "data-source" / "data",
    Path.cwd().parent / "data-source" / "data",
]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Fant ikke data-source/data")

CSV_PATH = DATA_DIR / "cleaned_merged_seasons_team_aggregated_expanded.csv"
df = pd.read_csv(CSV_PATH, low_memory=False)
df["kickoff_time"] = pd.to_datetime(df["kickoff_time"], errors="coerce", utc=True)

numeric_columns = [
    "GW", "minutes", "total_points", "goals_scored", "assists",
    "clean_sheets", "goals_conceded", "bonus", "bps", "saves",
    "yellow_cards", "red_cards", "creativity", "influence",
    "threat", "ict_index", "value", "selected", "transfers_balance",
    "points", "team_goals_scored", "team_goals_conceded", "team_goals_diff",
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

print(f"Lastet {len(df):,} rader fra {CSV_PATH.name}")
print("Tilgjengelige sesonger:", sorted(df["season_x"].dropna().unique()))

## Velg sesong og spiller

In [ ]:
seasons = sorted(df["season_x"].dropna().unique(), reverse=True)
season_dropdown = widgets.Dropdown(options=seasons, value=seasons[0], description="Sesong:")
player_dropdown = widgets.Dropdown(description="Spiller:", layout=widgets.Layout(width="400px"))
output = widgets.Output()

def players_in_season(season):
    return sorted(df.loc[df["season_x"].eq(season), "name"].dropna().unique())

def update_players(change=None):
    names = players_in_season(season_dropdown.value)
    player_dropdown.options = names
    if names:
        player_dropdown.value = names[0]

update_players()

In [ ]:
def show_player(season, player):
    with output:
        clear_output(wait=True)
        if not player:
            print("Ingen spiller valgt.")
            return

        player_data = (
            df.loc[df["season_x"].eq(season) & df["name"].eq(player)]
            .sort_values(["GW", "kickoff_time", "fixture"])
            .copy()
        )
        if player_data.empty:
            print("Fant ingen data.")
            return

        player_data["price_m"] = player_data["value"] / 10
        played = player_data.loc[player_data["minutes"].fillna(0).gt(0)]
        minutes = player_data["minutes"].sum()
        position = player_data["position"].dropna().iloc[0] if player_data["position"].notna().any() else "Ukjent"
        teams = ", ".join(player_data["team"].dropna().astype(str).loc[lambda x: x.ne("")].unique()) or "Mangler i filen"

        summary = pd.DataFrame({
            "Verdi": [
                position, teams, len(player_data), len(played), int(minutes),
                int(player_data["total_points"].sum()),
                int(player_data["goals_scored"].sum()),
                int(player_data["assists"].sum()),
                int(player_data["clean_sheets"].sum()),
                int(player_data["bonus"].sum()),
                round(player_data["total_points"].sum() / minutes * 90, 2) if minutes else None,
                round(player_data["ict_index"].sum() / minutes * 90, 2) if minutes else None,
            ]
        }, index=[
            "Posisjon", "Lag", "Registrerte kamper", "Kamper med minutter", "Minutter",
            "FPL-poeng", "Mål", "Assists", "Clean sheets", "Bonus",
            "Poeng per 90", "ICT per 90",
        ])

        print(f"{player} – {season}")
        display(summary)

        gw_data = (
            player_data.groupby("GW", as_index=False)
            .agg(
                total_points=("total_points", "sum"), minutes=("minutes", "sum"),
                goals=("goals_scored", "sum"), assists=("assists", "sum"),
                bonus=("bonus", "sum"), ict=("ict_index", "sum"),
                creativity=("creativity", "sum"), influence=("influence", "sum"),
                threat=("threat", "sum"), price_m=("price_m", "last"),
            )
        )
        gw_data["cumulative_points"] = gw_data["total_points"].cumsum()

        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        axes[0, 0].bar(gw_data["GW"], gw_data["total_points"], color="#db0007")
        axes[0, 0].set(title="FPL-poeng per GW", xlabel="GW", ylabel="Poeng")
        axes[0, 1].plot(gw_data["GW"], gw_data["cumulative_points"], marker="o", color="#063672")
        axes[0, 1].set(title="Akkumulerte poeng", xlabel="GW", ylabel="Poeng")
        axes[1, 0].bar(gw_data["GW"], gw_data["minutes"], color="#9c824a")
        axes[1, 0].axhline(90, color="black", linewidth=1, linestyle="--")
        axes[1, 0].set(title="Minutter per GW", xlabel="GW", ylabel="Minutter")
        axes[1, 1].plot(gw_data["GW"], gw_data["creativity"], label="Creativity")
        axes[1, 1].plot(gw_data["GW"], gw_data["influence"], label="Influence")
        axes[1, 1].plot(gw_data["GW"], gw_data["threat"], label="Threat")
        axes[1, 1].set(title="ICT-komponenter", xlabel="GW")
        axes[1, 1].legend()
        for axis in axes.flat:
            axis.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()

        columns = [
            "GW", "kickoff_time", "opp_team_name", "was_home", "minutes",
            "goals_scored", "assists", "clean_sheets", "bonus", "bps",
            "creativity", "influence", "threat", "ict_index",
            "total_points", "price_m", "selected", "transfers_balance",
        ]
        print("Kamp-for-kamp")
        display(player_data[columns].reset_index(drop=True))

def refresh(change=None):
    show_player(season_dropdown.value, player_dropdown.value)

season_dropdown.observe(lambda change: (update_players(), refresh()), names="value")
player_dropdown.observe(refresh, names="value")
display(widgets.HBox([season_dropdown, player_dropdown]), output)
refresh()